In [11]:
import json
import pandas as pd
import certifi
from pathlib import Path
from neo4j import GraphDatabase, TrustCustomCAs

TRANSFORM_DIR = Path.cwd().parent / "Transform"

NEO4J_URI      = "neo4j://78976b17.databases.neo4j.io"
NEO4J_USER     = "78976b17"
NEO4J_PASSWORD = "RbsKnY-5Wf0aTTdOfqC60d-dTlVVf-LzTpXmvRcfJXc"
NEO4J_DATABASE = "78976b17"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    encrypted=True,
    trusted_certificates=TrustCustomCAs(certifi.where()),
)
driver.verify_connectivity()
print(f"Connesso a {NEO4J_URI}")

def run_write(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_write(lambda tx: tx.run(cypher, **params).consume().counters)

def run_read(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

Connesso a neo4j://78976b17.databases.neo4j.io


In [12]:
INPUT_JSON      = TRANSFORM_DIR / "strutture_merged.json"
INPUT_RESIDENTI = TRANSFORM_DIR / "residenti_lombardia_per_fascia.csv"

with open(INPUT_JSON, encoding="utf-8") as f:
    docs = json.load(f)

for doc in docs:
    doc["Rating_dimensioni"] = doc.get("Rating_dimensioni") or {}
    doc["citta_match"]       = (doc.get("Città") or "").upper()

hub        = [d for d in docs if d.get("Classificazione") == "HUB"]
spoke      = [d for d in docs if d.get("Classificazione") == "SPOKE"]
case_cdc   = [d for d in docs if d.get("Classificazione") == "CASA DI COMUNITÀ"]

print(f"Hub: {len(hub)} | Spoke: {len(spoke)} | Case di comunità: {len(case_cdc)}")

Hub: 21 | Spoke: 113 | Case di comunità: 193


In [13]:
SIGLA = {
    "Bergamo": "BG", "Brescia": "BS", "Como": "CO", "Cremona": "CR",
    "Lecco": "LC", "Lodi": "LO", "Mantova": "MN", "Milano": "MI",
    "Monza e Brianza": "MB", "Pavia": "PV", "Sondrio": "SO", "Varese": "VA",
}

def fascia_key(fascia: str) -> str:
    return fascia.replace("-", "_").replace("+", "plus")

res = pd.read_csv(INPUT_RESIDENTI, sep=";", dtype=str)
res["Maschi"]  = pd.to_numeric(res["Maschi"],  errors="coerce").fillna(0).astype(int)
res["Femmine"] = pd.to_numeric(res["Femmine"], errors="coerce").fillna(0).astype(int)
res["Totale"]  = pd.to_numeric(res["Totale"],  errors="coerce").fillna(0).astype(int)

comuni_docs = []
for (codice, provincia, comune), grp in res.groupby(["Codice comune", "Provincia", "Comune"]):
    doc_comune = {
        "Nome":            comune.upper(),
        "Provincia":       SIGLA.get(provincia, provincia),
        "Totale_abitanti": int(grp["Totale"].sum()),
    }
    for _, row in grp.iterrows():
        k = fascia_key(row["Fascia"])
        doc_comune[f"Maschi_{k}"]  = int(row["Maschi"])
        doc_comune[f"Femmine_{k}"] = int(row["Femmine"])
        doc_comune[f"Totale_{k}"]  = int(row["Totale"])
    comuni_docs.append(doc_comune)

print(f"Comuni: {len(comuni_docs)}")

Comuni: 1502


In [14]:
run_write("MATCH (n) DETACH DELETE n")

SummaryCounters({'contains-updates': True, 'nodes-deleted': 1926, 'relationships-deleted': 326})

In [15]:
run_write("CREATE CONSTRAINT struttura_nome IF NOT EXISTS FOR (s:Struttura) REQUIRE s.Nome IS UNIQUE")
run_write("CREATE CONSTRAINT area_nome      IF NOT EXISTS FOR (a:AreaSpecialistica) REQUIRE a.Nome IS UNIQUE")
run_write("CREATE CONSTRAINT comune_nome    IF NOT EXISTS FOR (c:Comune) REQUIRE c.Nome IS UNIQUE")

SummaryCounters({})

In [16]:
run_write("""
UNWIND $docs AS row
MERGE (c:Comune {Nome: row.Nome})
SET c += row
""", docs=comuni_docs)

SummaryCounters({'contains-updates': True, 'labels-added': 1502, 'nodes-created': 1502, 'properties-set': 24032})

In [17]:
STRUTTURA_PROPS = """
SET
  s.Indirizzo        = row.Indirizzo,
  s.Ente             = row.Ente,
  s.Classificazione  = row.Classificazione,
  s.Rating_globale   = row.Rating_globale,
  s.N_valutazioni    = row.N_valutazioni,
  s.N_aree           = row.N_aree,
  s.Coordinate       = CASE
      WHEN row.Latitudine IS NOT NULL AND row.Longitudine IS NOT NULL
      THEN point({latitude: row.Latitudine, longitude: row.Longitudine})
      ELSE null
    END
SET s += row.Rating_dimensioni
"""

run_write(
    f"UNWIND $docs AS row MERGE (s:Struttura:Ospedale:Hub   {{Nome: row.Nome}}) {STRUTTURA_PROPS}",
    docs=hub
)

run_write(
    f"UNWIND $docs AS row MERGE (s:Struttura:Ospedale:Spoke {{Nome: row.Nome}}) {STRUTTURA_PROPS}",
    docs=spoke
)

run_write(
    f"UNWIND $docs AS row MERGE (s:Struttura:CasaDiComunita {{Nome: row.Nome}}) {STRUTTURA_PROPS}",
    docs=case_cdc
)

SummaryCounters({'contains-updates': True, 'labels-added': 386, 'nodes-created': 193, 'properties-set': 1544})

In [18]:
aree_uniche = sorted({
    area["Nome"]
    for doc in docs
    for area in doc.get("Aree_specialistiche", [])
    if area.get("Nome")
})
print(f"Aree uniche: {len(aree_uniche)}")

run_write("UNWIND $nomi AS nome MERGE (:AreaSpecialistica {Nome: nome})", nomi=aree_uniche)

Aree uniche: 97


SummaryCounters({'contains-updates': True, 'labels-added': 97, 'nodes-created': 97, 'properties-set': 97})

In [20]:
run_write("""
UNWIND $docs AS row
MATCH (s:Struttura {Nome: row.Nome})
MATCH (c:Comune    {Nome: row.citta_match})
MERGE (c)-[:HA_STRUTTURA]->(s)
""", docs=docs)

SummaryCounters({})

In [21]:
run_write("""
UNWIND $docs AS row
MATCH (s:Struttura {Nome: row.Nome})
UNWIND row.Aree_specialistiche AS area
MATCH (a:AreaSpecialistica {Nome: area.Nome})
MERGE (s)-[r:HA_AREA]->(a)
SET r.Rating = area.Rating
""", docs=docs)

SummaryCounters({'contains-updates': True, 'relationships-created': 3478, 'properties-set': 418})

In [22]:
for label in ["Comune", "Struttura", "Hub", "Spoke", "CasaDiComunita", "AreaSpecialistica"]:
    n = run_read(f"MATCH (n:{label}) RETURN count(n) AS n")["n"].iloc[0]
    print(f"  :{label:<22} → {n} nodi")

for rel in ["HA_STRUTTURA", "HA_AREA"]:
    n = run_read(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n")["n"].iloc[0]
    print(f"  :{rel:<22} → {n} relazioni")

  :Comune                 → 1502 nodi
  :Struttura              → 327 nodi
  :Hub                    → 21 nodi
  :Spoke                  → 113 nodi
  :CasaDiComunita         → 193 nodi
  :AreaSpecialistica      → 97 nodi
  :HA_STRUTTURA           → 327 relazioni
  :HA_AREA                → 3478 relazioni
